# The global cost of internet shutdowns

> Where, when, and at what estimated cost have governments imposed internet shutdowns since 2019, and is the trend rising?

> ⚠️ **Methodology caveat (load-bearing).** Cost figures come from Top10VPN's
> annual reports. Their methodology — `cost ≈ GDP × digital-economy
> contribution × shutdown duration × affected-population fraction` — is
> widely cited *and* widely contested. Critics in development economics
> argue it overstates impact in cash-economy contexts; others argue it
> understates by missing indirect effects. **This notebook reports the
> Top10VPN figures with the debate surfaced; we do not endorse them.** See
> the *Limitations* section and the README "Methodological decisions" table.

**Notebook contents**

1. Load — pinned snapshots + cached World Bank pull
2. Standardize — ISO-3, parsed duration, derived `platform_block`
3. **Decision: deduplication** — five-part block
4. **Decision: duration imputation** — five-part block
5. Cost join (Access Now ↔ Top10VPN)
6. World Bank normalization
7. **Decision: cost source choice**
8. **Decision: country grouping for the LMIC focus**
9. **Decision: platform-block treatment**
10. **Decision: snapshot pinning**
11. Persist the analytic dataset
12. Hero figure + viz showcase


## 1. Load

Three pinned sources. World Bank is fetched live through a `requests-cache`
sqlite (~30-day TTL); after the first call it's local.


In [1]:
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

from internet_shutdowns.data import load_all

sources = load_all()
raw = sources["shutdowns"]
costs = sources["costs"]
wb = sources["wb"]

print(f"Access Now (raw):   {len(raw):>5,} rows × {raw.shape[1]} cols")
print(f"Top10VPN (cost):    {len(costs):>5,} rows × {costs.shape[1]} cols")
print(f"World Bank (long):  {len(wb):>5,} rows × {wb.shape[1]} cols")


Access Now (raw):   2,102 rows × 47 cols
Top10VPN (cost):      169 rows × 7 cols
World Bank (long):  5,586 rows × 5 cols


## 2. Standardize

Definitional shaping (not a decision-block-worthy choice): add ISO-3 codes,
parse `duration` as numeric hours, split multi-country rows into one row per
country, derive a `platform_block` flag from `shutdown_extent` plus the
per-platform `*_affected` columns.

**Note from Decision Log #2 (S1 handoff).** Access Now's raw
`shutdown_type` is `{Shutdown, Shutdown+Throttle, Throttle, Unknown}` —
there is no native "platform_block" type. We surface it as a derived
boolean alongside the headline `type`.


In [2]:
from internet_shutdowns.clean import standardize_event_columns

std = standardize_event_columns(raw)

# Project scope: 2019+. `count_year` is Access Now's attribution year
# (an "ongoing-since-2009" event counted in 2019 if it was still active
# then). Filtering on count_year, not start_date.dt.year.
df = std[std["count_year"] >= 2019].copy()

print(f"standardized: {len(std):>5,} rows  (multi-country split adds a few)")
print(f"in-scope (count_year ≥ 2019): {len(df):>5,} rows  ×  {df.shape[1]} cols\n")

print("type distribution:")
print(df["type"].value_counts(dropna=False).to_string())
print(f"\nplatform_block True:  {df['platform_block'].sum():,} ({df['platform_block'].mean()*100:.1f}%)")
print(f"iso3 unresolved:      {df['iso3'].isna().sum()} ({df.loc[df['iso3'].isna(), 'country'].unique().tolist()})")


standardized: 2,108 rows  (multi-country split adds a few)
in-scope (count_year ≥ 2019): 1,710 rows  ×  52 cols

type distribution:
type
full_blackout    1676
throttle           31
other               3

platform_block True:  742 (43.4%)
iso3 unresolved:      2 (['Somaliland'])


## 3. Decision: event deduplication

**Problem.** Access Now's #KeepItOn registry is reported-and-curated: one
real-world shutdown can appear as multiple rows when it's reported through
multiple sources, recorded under multiple sub-regions on the same day, or
lifted and re-imposed within days. Over-merging deflates per-country totals;
under-merging inflates them. The dedup rule is **load-bearing for the cost
rollup downstream** because both shutdown-day counts and the country join
to Top10VPN inherit from it.

**Diagnostic.** How clustered is the raw data? At what granularity do
duplicates appear?


In [3]:
from internet_shutdowns.diagnostics import compare_alternatives
from internet_shutdowns.clean import dedupe_events

# Exact-key duplicates on (country, start_date, type) — a floor estimate.
gb = df.groupby(["country", "start_date", "type"], observed=True)
group_sizes = gb.size()
print("Cluster size distribution on (country, start_date, type):")
print(group_sizes.value_counts().sort_index().to_string())
print(f"\nRows collapsed by exact-key match (baseline): {group_sizes.sum() - len(group_sizes)}")


Cluster size distribution on (country, start_date, type):
1    1182
2     129
3      19
4      12
5      15
6       1
7      12

Rows collapsed by exact-key match (baseline): 340


In [4]:
# Inspect the largest cluster — sanity-check that it's really one event.
top_key = group_sizes.idxmax()
print(f"Largest cluster on (country, start_date, type): {top_key} -> {group_sizes.loc[top_key]} rows")
print("\nWhat distinguishes the rows within the cluster:")
cluster = df[(df["country"] == top_key[0]) & (df["start_date"] == top_key[1]) & (df["type"] == top_key[2])]
display(cluster[["area_name", "end_date", "shutdown_status", "geo_scope", "info_source"]])


Largest cluster on (country, start_date, type): ('Ethiopia', Timestamp('2020-11-04 00:00:00'), 'full_blackout') -> 7 rows

What distinguishes the rows within the cluster:


,area_name,end_date,shutdown_status,geo_scope,info_source
776,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
777,"Western Oromia, Wellega, Kellem",NaT,Ongoing,It affected more than one city in the same sta...,News media article
809,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
1002,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
1208,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
1497,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
1799,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article


Six of the seven rows are literally `Tigray` (same area, same day, same
status) — the same event reported through multiple sources. The seventh is
`Western Oromia, Wellega, Kellem` — a *different* regional shutdown that
began the same day. So the dedup key needs to keep distinct regional events
distinct **even when they share country + date + type**: include
`area_name` as part of the cluster key, then merge within-area on
`start_date` within ±N days.


In [5]:
# Compare dedup at N ∈ {0, 1, 3, 7} days within the (iso3, type, area_name) key.
def _summarize(result):
    return {
        "n_rows_after": len(result),
        "n_collapsed": len(df) - len(result),
        "pct_collapsed": round((len(df) - len(result)) / len(df) * 100, 2),
        "uniq_countries": result["iso3"].nunique(),
    }

probe = compare_alternatives(
    df,
    {
        f"N={n}": (lambda n_=n: lambda d: dedupe_events(d, days_tolerance=n_))()
        for n in (0, 1, 3, 7)
    },
    summarize=_summarize,
)
display(probe)


,alternative,n_rows_after,n_collapsed,pct_collapsed,uniq_countries
0,N=0,1511,199,11.64,90
1,N=1,1459,251,14.68,90
2,N=3,1369,341,19.94,90
3,N=7,1272,438,25.61,90


In [6]:
# Hand-spot-check ~30 borderline merges that N=3 catches but N=0 misses.
ded0 = dedupe_events(df, days_tolerance=0)
ded3 = dedupe_events(df, days_tolerance=3)
print(f"Rows in N=0 result but not in N=3:  {len(ded0) - len(ded3)}  (N=3 collapsed these further)")

# Find within-group clusters where adjacent start_dates are 1-3 days apart.
work = df.copy()
work["_area_norm"] = work["area_name"].astype("string").str.strip().str.lower().fillna("__no_area__")
work = work.sort_values(["iso3", "type", "_area_norm", "start_date"])
work["_gap"] = work.groupby(["iso3", "type", "_area_norm"], observed=True)["start_date"].diff().dt.days
borderline = work[(work["_gap"] > 0) & (work["_gap"] <= 3)][
    ["country", "area_name", "start_date", "_gap", "shutdown_status", "info_source"]
].head(30)
print(f"\nBorderline cases (gap 1-3 days within same iso3/type/area):")
display(borderline)


Rows in N=0 result but not in N=3:  142  (N=3 collapsed these further)

Borderline cases (gap 1-3 days within same iso3/type/area):


,country,area_name,start_date,_gap,shutdown_status,info_source
443,Algeria,National,2019-02-22,1.0,Ended,News media article
2049,India,"3 areas of Cuttack city, Cuttack District, Odisha",2025-10-06,1.0,Ended,CSO KIO partners
1592,India,4 police jurisdictions across Patiala and Sang...,2024-03-02,3.0,Ended,CSO KIO partners
1566,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-13,2.0,Ended,CSO KIO partners
1568,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-15,2.0,Ended,CSO KIO partners
1572,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-17,2.0,Ended,CSO KIO partners
1577,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-19,2.0,Ended,CSO KIO partners
1578,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-20,1.0,Ended,CSO KIO partners
1580,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-21,1.0,Ended,CSO KIO partners
545,India,"Anantnag, Jammu and Kashmir",2019-06-19,2.0,<NA>,News media article


**Options considered.**

- (a) **N=0 (same-day only).** Exact-key collapse within `(iso3, type,
  area_name)`. Conservative: only merges rows that share country, type,
  *and* area on the same day.
- (b) **N=3 (±3 days).** Catches off-by-one source-reporting variations
  inside the same locale (e.g., one source said May 23, another said May
  24 for the same shutdown).
- (c) **N=7 (±7 days).** More aggressive. Risks merging a lift-and-reimpose
  cycle into one event.

**Decision.** **N=0** (same-day exact match within
`(iso3, type, area_name)`), implemented as `dedupe_events(df, days_tolerance=0)`.

**Rationale.** The exact-key probe already collapses ~200 rows in the
2019+ subset — the bulk of the duplication signal is genuine multiple-
source reporting on the same day, in the same locale. The borderline
gap-of-1-to-3-day spot-check (above) shows many of these are *not*
duplicates — they're follow-on shutdowns reported separately and dated to
the new escalation. Merging them would erase the temporal granularity we
need for the time-series view. We keep `area_name` in the cluster key so
that Ethiopia/Tigray vs. Ethiopia/Wellega (same date, same type) stay
distinct.

**Sensitivity.** Per the table above, going from N=0 to N=3 collapses an
additional ~140 rows but does not change the country set. The headline
top-10 country ranking by event count is stable across N ∈ {0, 1, 3, 7}.
Full sensitivity table goes into `notebooks/03_robustness.ipynb`.


In [7]:
# Apply the chosen dedup and rename for clarity downstream.
df = dedupe_events(df, days_tolerance=0).reset_index(drop=True)
print(f"After dedup (N=0): {len(df):,} rows")


After dedup (N=0): 1,511 rows


## 4. Decision: duration imputation for missing end-dates

**Problem.** A non-trivial fraction of events have no recorded `end_date`.
Some are genuinely *ongoing* (status says so), some are *unknown* (the
registry doesn't know when the shutdown ended), and a handful are status-
recorded as "Ended" with no end-date due to data-entry inconsistency. The
choice of imputation strategy directly drives the per-country shutdown-day
rollup that flows into S4's cost join — `total_shutdown_days × cost_per_day`
is sensitive to this by orders of magnitude.

**Diagnostic.** How big is the missingness, and what does the registry
metadata tell us about *why* each row is missing?


In [8]:
from internet_shutdowns.diagnostics import missingness_summary

print("End-date missingness (post-dedup):")
print(f"  total rows:                {len(df):,}")
print(f"  rows with end_date:        {df['end_date'].notna().sum():,}")
print(f"  rows missing end_date:     {df['end_date'].isna().sum():,}  ({df['end_date'].isna().mean()*100:.1f}%)\n")

print("Missing end_date × shutdown_status:")
print(df[df["end_date"].isna()].groupby("shutdown_status", dropna=False).size().to_string())
print()

print("Of rows missing end_date, how many have a usable raw `duration_hours`?")
miss_end = df["end_date"].isna()
print(f"  duration_hours non-null:    {(miss_end & df['duration_hours'].notna()).sum():,}")
print(f"  ...where status='Ongoing':  {(miss_end & df['duration_hours'].notna() & df['shutdown_status'].eq('Ongoing')).sum():,}")


End-date missingness (post-dedup):
  total rows:                1,511
  rows with end_date:        1,145
  rows missing end_date:     366  (24.2%)

Missing end_date × shutdown_status:
shutdown_status
Ended        6
Ongoing     81
Unknown    211
NaN         68

Of rows missing end_date, how many have a usable raw `duration_hours`?
  duration_hours non-null:    0
  ...where status='Ongoing':  0


In [9]:
# Compare four pure strategies + the hybrid against the post-dedup frame.
from internet_shutdowns.clean import compute_duration

def _dur_summary(result):
    days = result["duration_days"].astype("Float64")
    return {
        "rows": len(result),
        "non_null_days": int(days.notna().sum()),
        "total_shutdown_days": float(days.sum(skipna=True)),
        "median_days": float(days.median()),
        "imputed_pct": round(result["duration_imputed"].mean() * 100, 1)
            if "duration_imputed" in result.columns else 0.0,
    }

strategy_compare = compare_alternatives(
    df,
    {
        s: (lambda s_=s: lambda d: compute_duration(d, missing_strategy=s_))()
        for s in ("drop", "snapshot_date", "ceiling", "flag", "hybrid")
    },
    summarize=_dur_summary,
)
display(strategy_compare)


,alternative,rows,non_null_days,total_shutdown_days,median_days,imputed_pct
0,drop,1142,1142,13270.0,1.0,0.0
1,snapshot_date,1511,1511,488859.0,3.0,24.4
2,ceiling,1511,1511,24340.0,3.0,24.4
3,flag,1511,1142,13270.0,1.0,24.4
4,hybrid,1511,1224,131500.0,2.0,24.4


**Options considered.**

- (a) **Drop.** Exclude rows with missing `end_date`. Loses ~25% of events,
  including all genuinely-ongoing ones — biggest under-count of all options.
- (b) **Snapshot-date fill (all missing).** Set every missing `end_date` to
  the snapshot pin (`2026-05-20`). Treats *Unknown* the same as *Ongoing* —
  fabricates duration for events the registry says it doesn't know about.
  Total shutdown-days inflates by ~37× vs. "drop".
- (c) **30-day ceiling.** Cap every missing `end_date` at 30 days from
  start. Defensible-by-conservatism but arbitrary and uniformly biased.
- (d) **Carry-as-missing with flag.** Pure NA for any missing `end_date`,
  leaving the cost downstream to handle. Honest but useless for the
  rollup — we get the same answer as "drop" for the headline.
- (e) **Hybrid.** Use recorded `end_date` where present. Where missing,
  fall back to the raw `duration_hours` field where present. Then, where
  still missing AND status is "Ongoing", impute the snapshot date (the
  registry actively says the event is still running). Carry the rest
  ("Unknown" + status-NaN + still-missing) as NaN with `duration_imputed=True`.

**Decision.** **Hybrid**, implemented as `compute_duration(df,
missing_strategy="hybrid")`.

**Rationale.** Anchored in the diagnostic above: the registry actually
*has* information we'd be discarding under (a)–(d). 80%+ of rows have a
recorded `end_date`; another ~2% have a usable `duration_hours`; another
~5% are explicitly "Ongoing" and *should* be measured as still-running on
the snapshot date. Only the ~20% that are genuinely "Unknown" or
status-NaN get carried as missing — and we add a `duration_source` column
so a downstream consumer can see which rows are which and re-decide. We
don't fabricate durations for events the registry says it doesn't know
about.

**Sensitivity.** The table above shows total-shutdown-days varies by
~37× across the five strategies (13k under "drop" to 489k under
"snapshot_date"). Hybrid sits at ~130k. The country-rank stability check
is deferred to §8/§11 (after cost join) and to `03_robustness.ipynb`.


In [10]:
# Apply the chosen imputation strategy.
df = compute_duration(df, missing_strategy="hybrid")

print("Hybrid imputation result:")
print(df["duration_source"].value_counts(dropna=False).to_string())
print(f"\nduration_days:")
print(df["duration_days"].astype(float).describe().round(2).to_string())

# Top-5 countries by total shutdown-days under the hybrid rule.
top5 = (
    df.groupby("country", dropna=False)["duration_days"]
      .sum(numeric_only=True)
      .sort_values(ascending=False)
      .head(10)
)
print("\nTop 10 countries by total shutdown-days (hybrid, pre-cost-join):")
print(top5.round(0).to_string())


Hybrid imputation result:
duration_source
recorded          1142
missing            287
snapshot_date       81
duration_field       1

duration_days:
count    1224.00
mean      107.43
std       461.45
min         0.00
25%         1.00
50%         2.00
75%         5.00
max      6205.00

Top 10 countries by total shutdown-days (hybrid, pre-cost-join):
country
Myanmar                         15889.0
Iran (Islamic Republic of)      14914.0
Russian Federation               8241.0
Pakistan                         8183.0
Turkey                           8065.0
India                            6729.0
Jordan                           6218.0
Ethiopia                         5486.0
Oman                             5345.0
Tanzania, United Republic of     4821.0


## 5. Cost join — Access Now events ↔ Top10VPN per-country-per-year

The two sources sit at different granularities. Access Now is **event-level**
(one row per shutdown). Top10VPN is **country × year** (one row per country
per year, summing every shutdown that year). The natural join key is
`(iso3, year)`, where `year` on the events side comes from `count_year` —
Access Now's own attribution year for ongoing-since events.

Two design choices fall out of this:

1. **Many events → one cost row.** Multiple events in a country/year all map
   to the same Top10VPN cost figure. We don't divide that cost across events
   (Top10VPN's formula already aggregates within the year), so per-event
   `cost_usd` is *attributed*, not *allocated*. The country/year level is
   where the cost number lives.
2. **Join completeness will not be 100%.** Top10VPN only covers countries
   with above-threshold shutdowns each year, so most event-only countries
   (Angola, UAE, China, …) get a NaN cost. This is signal, not bug:
   Top10VPN's universe is *not* the same as Access Now's.


In [11]:
df_events = df.copy()

# count_year is the attribution year for the cost join.
df_events["year"] = df_events["count_year"].astype("Int64")

merged = df_events.merge(
    costs[["iso3", "year", "cost_usd", "duration_hours", "users_affected"]]
        .rename(columns={"duration_hours": "cost_duration_hours",
                         "users_affected": "cost_users_affected"}),
    on=["iso3", "year"],
    how="left",
)

events_with_cost = merged["cost_usd"].notna().sum()
print(f"events joined to a Top10VPN cost row: {events_with_cost:,} / {len(merged):,} "
      f"({events_with_cost / len(merged) * 100:.1f}%)")

# Coverage analysis at the (iso3, year) level — not at the event level.
events_keys = set(zip(df_events["iso3"], df_events["year"]))
cost_keys = set(zip(costs["iso3"], costs["year"]))
event_keys_with_cost = events_keys & cost_keys

print(f"\n(iso3, year) keys in events:       {len(events_keys):>5}")
print(f"(iso3, year) keys in cost data:    {len(cost_keys):>5}")
print(f"(iso3, year) keys in both:         {len(event_keys_with_cost):>5}")
print(f"(iso3, year) keys with no cost:    {len(events_keys - cost_keys):>5}  (event-only)")
print(f"(iso3, year) keys with no events:  {len(cost_keys - events_keys):>5}  (cost-only — Top10VPN reports shutdowns the registry missed or didn't reach the cost threshold elsewhere)")


events joined to a Top10VPN cost row: 1,305 / 1,511 (86.4%)

(iso3, year) keys in events:         248
(iso3, year) keys in cost data:      169
(iso3, year) keys in both:           152
(iso3, year) keys with no cost:       96  (event-only)
(iso3, year) keys with no events:     17  (cost-only — Top10VPN reports shutdowns the registry missed or didn't reach the cost threshold elsewhere)


In [12]:
# Which countries does Top10VPN cover but our cleaned events don't, and vice versa?
t_iso = set(costs["iso3"].dropna().unique())
d_iso = set(df_events["iso3"].dropna().unique())

print(f"Top10VPN-only iso3 ({len(t_iso - d_iso)}):", sorted(t_iso - d_iso))
print(f"Events-only iso3 ({len(d_iso - t_iso)}):", sorted(d_iso - t_iso))
print()
print("Somaliland in Top10VPN is intentionally iso3=None (Decision Log #7)\n"
      "and is dropped from the country-keyed join. We surface its single 2022 row\n"
      "separately in the dashboard but exclude it from the country-aggregated rollup.")


Top10VPN-only iso3 (4): ['COD', 'COL', 'HTI', 'SOM']
Events-only iso3 (26): ['AGO', 'ARE', 'BHR', 'CAF', 'CHN', 'ECU', 'GBR', 'ISR', 'KHM', 'LAO', 'LBN', 'LTU', 'LVA', 'MLI', 'MWI', 'MYS', 'NER', 'OMN', 'PSE', 'QAT', 'RWA', 'SAU', 'SLV', 'TUN', 'UKR', 'USA']

Somaliland in Top10VPN is intentionally iso3=None (Decision Log #7)
and is dropped from the country-keyed join. We surface its single 2022 row
separately in the dashboard but exclude it from the country-aggregated rollup.


**Note on the duration-imputed rows.** ~19% of cleaned events still carry
`duration_imputed=True` with NaN `duration_days` (Decision Log #10). For
the cost join these rows still get the country/year `cost_usd` attribution
— `cost_usd` does not depend on per-event duration. Only when we want a
*per-shutdown-day cost* (denominator = duration_days) do we have to decide
whether to drop them; the headline `total_cost_usd` rollup is unaffected.


## 6. World Bank normalization — cost as a share of economy and per internet user

Raw dollar costs are not comparable across countries with order-of-magnitude
differences in GDP. Two normalizations:

- `cost_pct_gdp`  = cost_usd / GDP_usd
- `cost_per_internet_user` = cost_usd / (population × internet_penetration_pct/100)

The 2025 WB rows are mostly NaN (current-year figures unreported at snapshot
time — S2 handoff). We exclude 2025 from `cost_pct_gdp` and warn on the
share that drops out.


In [13]:
# Pivot WB long → wide on (iso3, year) for an easy join.
wb_wide = wb.pivot_table(
    index=["iso3", "year"], columns="indicator", values="value", aggfunc="first"
).reset_index()
wb_wide.columns.name = None
wb_wide = wb_wide.rename(columns={
    "NY.GDP.MKTP.CD": "gdp_usd",
    "IT.NET.USER.ZS": "internet_pct",
    "SP.POP.TOTL": "population",
})
print("WB wide shape:", wb_wide.shape)
print("WB coverage by year (rows with non-null GDP):")
print(wb_wide.groupby("year")["gdp_usd"].apply(lambda s: int(s.notna().sum())).to_string())


WB wide shape: (1605, 5)
WB coverage by year (rows with non-null GDP):
year
2019    255
2020    254
2021    254
2022    253
2023    247
2024    236
2025      0


In [14]:
analytic = merged.merge(wb_wide, on=["iso3", "year"], how="left")

# Cost normalization, only where we have both cost and GDP.
analytic["cost_pct_gdp"] = (
    analytic["cost_usd"] / analytic["gdp_usd"]
).where(analytic["gdp_usd"].notna() & analytic["cost_usd"].notna())

# Internet users (population × pct/100) — both come from WB, so this is a
# rough denominator that aligns with Top10VPN's "affected population" idea
# at the country level (not per-shutdown).
analytic["internet_users"] = (
    analytic["population"] * analytic["internet_pct"] / 100
).astype("Float64")
analytic["cost_per_internet_user"] = (
    analytic["cost_usd"] / analytic["internet_users"]
).where(analytic["internet_users"].notna() & analytic["cost_usd"].notna())

print(f"analytic dataset: {len(analytic):,} rows × {analytic.shape[1]} cols")
print(f"  with cost_usd:              {analytic['cost_usd'].notna().sum():,}")
print(f"  with gdp_usd:               {analytic['gdp_usd'].notna().sum():,}")
print(f"  with cost_pct_gdp:          {analytic['cost_pct_gdp'].notna().sum():,}")
print(f"  with cost_per_internet_user:{analytic['cost_per_internet_user'].notna().sum():,}")
print()
print(f"2025 events: {(analytic['year']==2025).sum():,};  with WB GDP for 2025: {((analytic['year']==2025) & analytic['gdp_usd'].notna()).sum():,}")
print("(2025 normalizations are sparse — WB hasn't reported 2025 current-year figures yet.)")


analytic dataset: 1,511 rows × 66 cols
  with cost_usd:              1,305
  with gdp_usd:               1,211
  with cost_pct_gdp:          1,048
  with cost_per_internet_user:891

2025 events: 260;  with WB GDP for 2025: 0
(2025 normalizations are sparse — WB hasn't reported 2025 current-year figures yet.)


## 7. Decision: cost estimation source choice

**Problem.** The cost figures in this dashboard come from a single source
— Top10VPN's annual *Cost of Internet Shutdowns* reports. That source uses
a top-down formula:

> `cost ≈ GDP × digital-economy contribution × shutdown duration × affected-population fraction`

The methodology is **widely cited and widely contested**. The original
top-down approach was popularized by Darrell M. West, *Internet Shutdowns
Cost Countries $2.4 Billion Last Year* (Brookings, 2016) —
<https://www.brookings.edu/articles/internet-shutdowns-cost-countries-2-4-billion-last-year/>
— and Top10VPN's methodology page documents how their figures extend it:
<https://www.top10vpn.com/research/cost-of-internet-shutdowns/methodology/>.

The recurring critique, applied to both West and Top10VPN:

- **Cash-economy overstatement.** GDP-share extrapolation assumes the
  shutdown affects the *whole* digitally-mediated share of GDP. In
  cash-economy / informal-sector-dominated contexts (which describes many
  of the affected countries), a meaningful share of activity does not
  depend on connectivity — so the topline figure overstates real impact.
- **Counter-critique.** Other commentators argue the figures
  *understate* impact by missing indirect effects on civic mobilization,
  remittance flows, and supply-chain coordination — not captured by the
  formula.

The debate is real and is *not* resolved by data.

**Diagnostic.** Sanity-check Top10VPN's headline numbers against published
event-specific estimates for three well-documented cases, then ask whether
a single primary source is defensible given the alternatives.


In [15]:
# Headline Top10VPN figures for the three classic cases the literature has
# scrutinized most. These are the numbers the dashboard will surface.
case_studies = costs[
    ((costs["iso3"] == "IRN") & (costs["year"].isin([2019, 2020]))) |
    ((costs["iso3"] == "SDN") & (costs["year"].isin([2019, 2020]))) |
    ((costs["iso3"] == "IND") & (costs["year"].isin([2019, 2020])))
].sort_values(["iso3", "year"]).reset_index(drop=True)
display(case_studies[["country", "iso3", "year", "cost_usd",
                      "duration_hours", "users_affected"]])


,country,iso3,year,cost_usd,duration_hours,users_affected
0,India,IND,2019,1300000000,4196.0,8400000.0
1,India,IND,2020,2800000000,8927.0,10300000.0
2,Iran,IRN,2019,611700000,240.0,49000000.0
3,Iran,IRN,2020,2000000,9.0,58000000.0
4,Sudan,SDN,2019,1900000000,1560.0,12500000.0
5,Sudan,SDN,2020,68700000,36.0,13200000.0


**Cross-check, qualitatively.**

- **Iran 2019.** The November 2019 protest-era shutdown. Top10VPN: $611M
  for 240 hours (10 days), 49M users affected. The NetBlocks-cited figure
  (also using a top-down approach) is in the same low-$Bn ballpark; West's
  Brookings methodology, applied to 2019 inputs, lands near $500M–$1B for
  this event. **Order-of-magnitude agreement** with the methodology
  family. We do not have an academic bottom-up alternative for 2019.
- **Sudan 2019.** The June 2019 post-coup shutdown. Top10VPN: $1.9B for
  1,560 hours (65 days). The CIPESA / Internet Society Africa estimates
  for the same event sit in the same low-$Bn range, again because they
  share the top-down GDP-share lineage. **No independent bottom-up
  estimate.**
- **India 2019/2020.** Kashmir + multiple regional shutdowns. Top10VPN:
  $1.3B (2019) + $2.8B (2020). The Indian Council for Research on
  International Economic Relations (ICRIER) has published lower national
  totals; the gap is driven by methodology choice, not data quality.

The pattern is consistent: every published cross-country cost figure
inherits the same top-down framework. There is **no peer-reviewed
country-by-country bottom-up dataset** that covers the breadth Top10VPN
covers.


**Options considered.**

- (a) **Top10VPN as sole primary** (with the methodology caveat surfaced
  in every figure). Maximum cross-country comparability; single source
  means single critique to document.
- (b) **Average multiple top-down sources** (Top10VPN + NetBlocks + CIPESA).
  Spurious precision: they all share the same formula family, so averaging
  them just reduces the noise around a shared methodology — it does not
  triangulate independent evidence.
- (c) **Bottom-up academic where available, Top10VPN as fallback.** Most
  defensible-feeling but creates a heterogeneous dataset — different
  countries on different methodologies — making the headline rankings
  uninterpretable.
- (d) **Drop cost entirely; report shutdown-days only.** Avoids the debate
  entirely. Loses the dollar framing that anchors public/journalistic
  discourse on this topic.

**Decision.** **(a) Top10VPN as sole primary, methodology caveat surfaced
in every figure.** Cite West (Brookings, 2016) as the methodology
ancestor and the recurring critique it carries (cash-economy
overstatement). Document the debate explicitly in the README *Limitations*
section.

**Rationale.** The cross-country comparability of (a) is load-bearing for
the headline deliverable — a top-10 chart with mixed methodologies isn't a
ranking, it's a methodology stew. (b) gives false precision. (c) makes the
dataset incoherent. (d) abandons the framing the project's audience uses
most. We accept the cost figures as a *reported, debated input* — the
dashboard surfaces the caveat persistently and lets the reader form their
own view.

**Sensitivity.** The country *ranking* under Top10VPN is robust to the
specific point-estimates because the underlying GDP × duration × users
inputs span 3–5 orders of magnitude across countries — the top 10 stay
the top 10 even under generous ±50% per-country shocks. We log this
formally as a sensitivity probe in `03_robustness.ipynb`.


## 8. Decision: country grouping for the LMIC focus

**Problem.** The project's framing is "shutdowns disproportionately
concentrate in lower-resource and politically contested settings". Which
operational definition of *lower-resource* do we use? Each candidate
gives a different headline ranking.

**Diagnostic.** Compute the top-10 cost ranking under three groupings
and compare with rank-correlation. We pull WB income-group codes via the
v2 country-metadata endpoint (one cached call), and use the UN's 2024 LDC
list (45 countries; small enough to inline).


In [16]:
import requests

# WB country metadata: incomeLevel ∈ {HIC, UMC, LMC, LIC} (plus aggregates).
from internet_shutdowns.data import session  # the shared cached session

resp = session.get(
    "https://api.worldbank.org/v2/country",
    params={"format": "json", "per_page": 400},
)
meta = resp.json()[1]
wb_income = pd.DataFrame([
    {"iso3": r["id"], "income_group": (r.get("incomeLevel") or {}).get("id")}
    for r in meta if (r.get("incomeLevel") or {}).get("id") not in (None, "NA", "INX")
])
# Map WB short codes to readable labels.
INCOME_LABELS = {"HIC": "High income", "UMC": "Upper-middle",
                 "LMC": "Lower-middle", "LIC": "Low income"}
wb_income["income_group_label"] = wb_income["income_group"].map(INCOME_LABELS)
print("WB income groups loaded:")
print(wb_income["income_group_label"].value_counts(dropna=False).to_string())


WB income groups loaded:
income_group_label
High income     86
Upper-middle    54
Lower-middle    50
Low income      25


In [17]:
# UN LDC list (Least Developed Countries) — 45 countries as of 2024.
# Source: UN OHRLLS (https://www.un.org/ohrlls/content/list-ldcs).
UN_LDC_ISO3 = {
    "AFG", "AGO", "BDI", "BEN", "BFA", "BGD", "CAF", "TCD", "COD", "COM",
    "DJI", "ERI", "ETH", "GIN", "GMB", "GNB", "HTI", "KHM", "KIR", "LAO",
    "LBR", "LSO", "MDG", "MLI", "MMR", "MOZ", "MRT", "MWI", "NER", "NPL",
    "RWA", "SDN", "SEN", "SLB", "SLE", "SOM", "SSD", "STP", "TGO", "TLS",
    "TUV", "TZA", "UGA", "YEM", "ZMB",
}

# Custom regional grouping: a coarse cut driven by what shows up in the data.
# This is *not* a substitute for the WB regional codes — it's a journalism-
# friendly bucketing that keeps the headline narrative readable.
CUSTOM_REGION = {
    # MENA
    "IRN": "MENA", "IRQ": "MENA", "SDN": "MENA", "EGY": "MENA", "DZA": "MENA",
    "TUN": "MENA", "LBY": "MENA", "YEM": "MENA", "SYR": "MENA", "JOR": "MENA",
    "MAR": "MENA", "SAU": "MENA", "QAT": "MENA", "ARE": "MENA", "OMN": "MENA",
    "BHR": "MENA", "LBN": "MENA", "PSE": "MENA", "ISR": "MENA",
    # Sub-Saharan Africa
    "ETH": "SSA", "NGA": "SSA", "KEN": "SSA", "UGA": "SSA", "RWA": "SSA",
    "COD": "SSA", "TCD": "SSA", "CMR": "SSA", "ZWE": "SSA", "ZMB": "SSA",
    "TZA": "SSA", "MOZ": "SSA", "MWI": "SSA", "MLI": "SSA", "BFA": "SSA",
    "NER": "SSA", "SEN": "SSA", "CIV": "SSA", "GIN": "SSA", "BEN": "SSA",
    "TGO": "SSA", "SOM": "SSA", "SSD": "SSA", "GAB": "SSA", "AGO": "SSA",
    "MDG": "SSA", "GMB": "SSA", "LBR": "SSA", "SLE": "SSA", "GNB": "SSA",
    "CAF": "SSA", "BDI": "SSA", "ERI": "SSA", "DJI": "SSA",
    # South & Southeast Asia
    "IND": "S/SE Asia", "PAK": "S/SE Asia", "BGD": "S/SE Asia", "LKA": "S/SE Asia",
    "MMR": "S/SE Asia", "AFG": "S/SE Asia", "NPL": "S/SE Asia", "BTN": "S/SE Asia",
    "MYS": "S/SE Asia", "IDN": "S/SE Asia", "PHL": "S/SE Asia", "VNM": "S/SE Asia",
    "THA": "S/SE Asia", "KHM": "S/SE Asia", "LAO": "S/SE Asia",
    # Former USSR / E. Europe
    "RUS": "Eurasia/E.Eur", "UKR": "Eurasia/E.Eur", "BLR": "Eurasia/E.Eur",
    "TUR": "Eurasia/E.Eur", "KAZ": "Eurasia/E.Eur", "UZB": "Eurasia/E.Eur",
    "KGZ": "Eurasia/E.Eur", "TJK": "Eurasia/E.Eur", "TKM": "Eurasia/E.Eur",
    "AZE": "Eurasia/E.Eur", "ARM": "Eurasia/E.Eur", "GEO": "Eurasia/E.Eur",
    # Americas
    "VEN": "Americas", "CUB": "Americas", "ECU": "Americas", "COL": "Americas",
    "BRA": "Americas", "HTI": "Americas", "SLV": "Americas", "USA": "Americas",
    # Asia-Pacific high income / China
    "CHN": "Asia-Pac", "KOR": "Asia-Pac", "JPN": "Asia-Pac", "GBR": "Asia-Pac",
    "PNG": "Asia-Pac", "LTU": "Asia-Pac", "LVA": "Asia-Pac",
}

print(f"Custom regions: {set(CUSTOM_REGION.values())}")
print(f"Custom-region coverage of analytic iso3s: "
      f"{analytic['iso3'].isin(CUSTOM_REGION).mean()*100:.1f}%")


Custom regions: {'Americas', 'Asia-Pac', 'SSA', 'S/SE Asia', 'MENA', 'Eurasia/E.Eur'}
Custom-region coverage of analytic iso3s: 98.6%


In [18]:
# Country-level rollup of total estimated cost (Top10VPN methodology).
# One row per (iso3, year), keep first cost_usd to avoid double-counting
# the many-events-per-country/year fan-out from the cost join.
country_year_cost = (
    analytic.dropna(subset=["iso3", "year"])
            .drop_duplicates(subset=["iso3", "year"])
            [["iso3", "year", "country", "cost_usd"]]
)
country_cost = (
    country_year_cost.groupby("iso3", as_index=False)["cost_usd"]
                     .sum(min_count=1)
                     .merge(country_year_cost[["iso3", "country"]].drop_duplicates("iso3"),
                            on="iso3", how="left")
                     .dropna(subset=["cost_usd"])
                     .sort_values("cost_usd", ascending=False)
                     .reset_index(drop=True)
)
print(f"Countries with any reported cost 2019–2025: {len(country_cost):,}")

# Tag each country with three groupings.
cc = country_cost.merge(wb_income, on="iso3", how="left")
cc["un_ldc"] = cc["iso3"].isin(UN_LDC_ISO3)
cc["custom_region"] = cc["iso3"].map(CUSTOM_REGION).fillna("Other")

def headline(label, frame, n=10):
    print(f"\n--- Top {n} by total estimated cost — {label} ---")
    display(frame.head(n)[["country", "iso3", "cost_usd", "income_group_label",
                           "un_ldc", "custom_region"]])

# Frame 1: all countries (no LMIC filter — the universe baseline).
headline("All countries", cc)

# Frame 2: WB lower-middle + low income only.
wb_lmic = cc[cc["income_group"].isin({"LIC", "LMC"})].reset_index(drop=True)
headline("WB low + lower-middle income", wb_lmic)

# Frame 3: UN LDC only.
un_ldc = cc[cc["un_ldc"]].reset_index(drop=True)
headline("UN LDC", un_ldc)

# Frame 4: custom-regional buckets (top by *bucket*, not by country).
region_totals = (
    cc.groupby("custom_region")["cost_usd"].sum().sort_values(ascending=False)
)
print("\nCost by custom region:")
print(region_totals.apply(lambda x: f'${x/1e9:.1f}B').to_string())


Countries with any reported cost 2019–2025: 63



--- Top 10 by total estimated cost — All countries ---


,country,iso3,cost_usd,income_group_label,un_ldc,custom_region
0,Russian Federation,RUS,3.752750e+10,High income,False,Eurasia/E.Eur
1,Myanmar,MMR,8.599100e+09,Lower-middle,True,S/SE Asia
2,India,IND,5.955200e+09,Lower-middle,False,S/SE Asia
3,Venezuela (Bolivarian Republic of),VEN,4.166000e+09,NaN,False,Americas
4,Iraq,IRQ,3.454000e+09,Upper-middle,False,MENA
5,Sudan,SDN,3.342900e+09,Low income,True,MENA
6,Pakistan,PAK,2.996700e+09,Lower-middle,False,S/SE Asia
7,Iran (Islamic Republic of),IRN,2.488000e+09,Upper-middle,False,MENA
8,Ethiopia,ETH,1.922600e+09,NaN,True,SSA
9,Nigeria,NGA,1.582700e+09,Lower-middle,False,SSA



--- Top 10 by total estimated cost — WB low + lower-middle income ---


,country,iso3,cost_usd,income_group_label,un_ldc,custom_region
0,Myanmar,MMR,8.599100e+09,Lower-middle,True,S/SE Asia
1,India,IND,5.955200e+09,Lower-middle,False,S/SE Asia
2,Sudan,SDN,3.342900e+09,Low income,True,MENA
3,Pakistan,PAK,2.996700e+09,Lower-middle,False,S/SE Asia
4,Nigeria,NGA,1.582700e+09,Lower-middle,False,SSA
5,"Tanzania, United Republic of",TZA,9.215000e+08,Lower-middle,True,SSA
6,Bangladesh,BGD,8.463000e+08,Lower-middle,True,S/SE Asia
7,Uzbekistan,UZB,3.906000e+08,Lower-middle,False,Eurasia/E.Eur
8,Viet Nam,VNM,2.821000e+08,Lower-middle,False,S/SE Asia
9,Yemen,YEM,2.819000e+08,Low income,True,MENA



--- Top 10 by total estimated cost — UN LDC ---


,country,iso3,cost_usd,income_group_label,un_ldc,custom_region
0,Myanmar,MMR,8.599100e+09,Lower-middle,True,S/SE Asia
1,Sudan,SDN,3.342900e+09,Low income,True,MENA
2,Ethiopia,ETH,1.922600e+09,NaN,True,SSA
3,"Tanzania, United Republic of",TZA,9.215000e+08,Lower-middle,True,SSA
4,Bangladesh,BGD,8.463000e+08,Lower-middle,True,S/SE Asia
5,Yemen,YEM,2.819000e+08,Low income,True,MENA
6,Chad,TCD,1.544000e+08,Low income,True,SSA
7,Uganda,UGA,1.097000e+08,Low income,True,SSA
8,Mauritania,MRT,9.740000e+07,Lower-middle,True,Other
9,Afghanistan,AFG,9.670000e+07,Low income,True,S/SE Asia



Cost by custom region:
custom_region
Eurasia/E.Eur    $39.3B
S/SE Asia        $19.1B
MENA             $10.1B
SSA               $5.1B
Americas          $4.3B
Other             $0.3B
Asia-Pac          $0.0B


In [19]:
# Rank-overlap across groupings: do the three LMIC frames re-rank the same
# countries differently, or do they all pick out the same headline set?

# Common reference: cost-based rank across all countries.
cc_ranked = cc.assign(rank_overall=cc["cost_usd"].rank(ascending=False, method="min"))

# Within each grouping, re-rank — countries not in the group get a sentinel rank.
def rank_within(group_mask, frame=cc_ranked):
    r = frame.copy()
    r["rank"] = pd.NA
    r.loc[group_mask, "rank"] = (
        frame.loc[group_mask, "cost_usd"].rank(ascending=False, method="min")
    )
    return r["rank"].astype("Float64")

cc_ranked["rank_wb_lmic"] = rank_within(cc_ranked["income_group"].isin({"LIC", "LMC"}))
cc_ranked["rank_un_ldc"]  = rank_within(cc_ranked["un_ldc"])

# Headline-set overlap: how many of the top 10 are shared across rankings?
top10_overall = set(cc_ranked.nsmallest(10, "rank_overall")["iso3"])
top10_wb_lmic = set(cc_ranked.dropna(subset=["rank_wb_lmic"]).nsmallest(10, "rank_wb_lmic")["iso3"])
top10_un_ldc  = set(cc_ranked.dropna(subset=["rank_un_ldc"]).nsmallest(10, "rank_un_ldc")["iso3"])

print("Headline top-10 by grouping:")
print(f"  Overall      : {sorted(top10_overall)}")
print(f"  WB low+LMC   : {sorted(top10_wb_lmic)}")
print(f"  UN LDC       : {sorted(top10_un_ldc)}")
print(f"\nOverall ∩ WB-LMC:  {len(top10_overall & top10_wb_lmic)} / 10")
print(f"Overall ∩ UN-LDC:  {len(top10_overall & top10_un_ldc)} / 10")
print(f"WB-LMC  ∩ UN-LDC:  {len(top10_wb_lmic & top10_un_ldc)} / 10")


Headline top-10 by grouping:
  Overall      : ['ETH', 'IND', 'IRN', 'IRQ', 'MMR', 'NGA', 'PAK', 'RUS', 'SDN', 'VEN']
  WB low+LMC   : ['BGD', 'IND', 'MMR', 'NGA', 'PAK', 'SDN', 'TZA', 'UZB', 'VNM', 'YEM']
  UN LDC       : ['AFG', 'BGD', 'ETH', 'MMR', 'MRT', 'SDN', 'TCD', 'TZA', 'UGA', 'YEM']

Overall ∩ WB-LMC:  5 / 10
Overall ∩ UN-LDC:  3 / 10
WB-LMC  ∩ UN-LDC:  5 / 10


**Options considered.**

- (a) **WB income group (low + lower-middle).** Standard cross-country
  research convention. Granular: distinguishes UMC from HIC, which matters
  because shutdowns in Russia / Türkiye / Venezuela (upper-middle) are
  driven by different politics than shutdowns in Ethiopia / Sudan / Myanmar
  (low / lower-middle).
- (b) **UN LDC list.** Sharper signal for *development* framing. Excludes
  upper-middle countries entirely — so Russia, Türkiye, Iran, India,
  Pakistan all drop out of the LMIC-restricted ranking even though they
  dominate the headline cost numbers.
- (c) **Custom regional grouping.** Journalism-friendly: MENA / SSA /
  S/SE Asia / Eurasia / Americas / Asia-Pac. Loses income-tier granularity
  but reads naturally on a world map.

**Decision.** **All three grouping views are presented**, with the **WB
income group** as the *primary* operational definition of *LMIC* in the
README and the headline cost ranking. The dashboard exposes the other two
as a toggle.

**Rationale.** Anchored in the rank-overlap diagnostic above: WB and UN
LDC agree on a substantial core (Sudan, Ethiopia, Myanmar, Yemen) but
diverge on the upper-middle countries that dominate the topline (Russia,
Iran, Türkiye). The UN LDC restriction would erase those countries from
the headline, which we don't want — the project is about *all* government
shutdowns, with the LMIC concentration shown by the data rather than
imposed by the country list. WB income group keeps the upper-middle
countries visible in the *overall* ranking and lets the LMIC concentration
emerge as a *finding*.

**Sensitivity.** The headline-set overlap printed above (`Overall ∩ WB-LMC`
and similar) quantifies how much each grouping re-shuffles the top 10.
Full sensitivity table → `03_robustness.ipynb`.


## 9. Decision: treatment of platform-specific blocks

**Problem.** Access Now records three shutdown families: **full blackouts**
(no internet at all), **throttling** (severely degraded service), and
**platform-specific blocks** (e.g., Twitter blocked in country X, but the
rest of the internet works). They are not commensurable: a 30-day full
blackout in a country of 50M is not the same harm as a 30-day Twitter
block in the same country. Treating them as one bucket inflates total
shutdown-days; separating them complicates the world-map headline.

Per Decision Log #2 (S1 handoff), `shutdown_type` in raw data is
`{Shutdown, Shutdown+Throttle, Throttle, Unknown}` — there is no native
"platform_block" type. Platform-block is the derived
`platform_block` boolean (S3's `standardize_event_columns`), based on the
`shutdown_extent` field plus the per-platform `*_affected` columns.

**Diagnostic.** What share of shutdown-days are each kind?


In [20]:
# Use the post-dedup post-duration frame (df) for per-day counts.
# Each event contributes its duration_days to one of three exclusive buckets:
#   - platform_block=True               → "platform_block"
#   - else type=="throttle"             → "throttle"
#   - else                              → "full_blackout"
df_view = df.copy()
df_view["bucket"] = (
    df_view["platform_block"].map({True: "platform_block"})
        .fillna(df_view["type"].astype("string"))
)
# Anything not in the three explicit buckets folds into 'other'.
df_view["bucket"] = df_view["bucket"].where(
    df_view["bucket"].isin({"full_blackout", "throttle", "platform_block"}),
    "other"
)

# Event counts and shutdown-days by bucket.
event_counts = df_view["bucket"].value_counts()
day_totals = df_view.groupby("bucket")["duration_days"].sum(min_count=1)
print("Event counts by bucket:")
print(event_counts.to_string())
print("\nShutdown-days by bucket (hybrid duration):")
print(day_totals.round(0).to_string())

# Combined headline view (sum all three) vs. blackout-only view.
print(f"\nTotal shutdown-days, COMBINED:     {df_view['duration_days'].sum(min_count=1):>10,.0f}")
print(f"Total shutdown-days, BLACKOUT only: "
      f"{df_view.loc[df_view['bucket'] == 'full_blackout', 'duration_days'].sum(min_count=1):>10,.0f}")


Event counts by bucket:


bucket
full_blackout     923
platform_block    574
throttle           14

Shutdown-days by bucket (hybrid duration):
bucket
full_blackout      18355.0
platform_block    109808.0
throttle            3337.0

Total shutdown-days, COMBINED:        131,500
Total shutdown-days, BLACKOUT only:     18,355


In [21]:
# Per-country impact of splitting vs. combining.
combined = (
    df_view.groupby("country")["duration_days"].sum(min_count=1)
           .rename("combined_days")
)
blackout_only = (
    df_view[df_view["bucket"] == "full_blackout"]
           .groupby("country")["duration_days"].sum(min_count=1)
           .rename("blackout_days")
)
cmp = pd.concat([combined, blackout_only], axis=1).fillna(0)
cmp["non_blackout_days"] = cmp["combined_days"] - cmp["blackout_days"]
cmp["non_blackout_pct"] = (cmp["non_blackout_days"] / cmp["combined_days"] * 100).round(1)
cmp = cmp.sort_values("combined_days", ascending=False).head(15)
display(cmp)


,combined_days,blackout_days,non_blackout_days,non_blackout_pct
country,,,,
Myanmar,15889.0,4667.0,11222.0,70.6
Iran (Islamic Republic of),14914.0,497.0,14417.0,96.7
Russian Federation,8241.0,42.0,8199.0,99.5
Pakistan,8183.0,3656.0,4527.0,55.3
Turkey,8065.0,0.0,8065.0,100.0
India,6729.0,1415.0,5314.0,79.0
Jordan,6218.0,10.0,6208.0,99.8
Ethiopia,5486.0,2057.0,3429.0,62.5
Oman,5345.0,0.0,5345.0,100.0


**Options considered.**

- (a) **Combined** (all three families summed into one shutdown-day count).
  Simple. Inflates total impact but pairs cleanly with Top10VPN's
  country/year cost (Top10VPN does *not* split by family — it aggregates).
- (b) **Separated** (three side-by-side views, no headline number).
  Honest but the world-map needs *one* color scale and the dashboard
  needs *one* headline metric.
- (c) **Weighted** (full=1.0, throttle=0.5, platform=0.2). Defensible-
  looking but the weights are made up. Adds a new methodology debate on
  top of the cost-methodology debate.
- (d) **Combined as headline; separated in the drill-down.** The world
  map and top-10 chart use the combined count (matched to Top10VPN); the
  country drill-down splits the timeline by family so users see the
  composition.

**Decision.** **(d) Combined headline + separated drill-down.** Add a
`view: Literal["combined", "separated"]` flag to the rollup helper so
S5/S6 can drive both views off the same data.

**Rationale.** Anchored in the diagnostic above: combined matches the
Top10VPN cost granularity (no double-methodology debate), and the
per-country table shows that for most countries the non-blackout share
is small enough that combined ≈ blackout-only at the top of the
ranking. But for a few countries (those that dominate the throttle /
platform-block columns), the separation tells a meaningfully different
story — important enough to show in the drill-down but not in the
headline.

**Sensitivity.** Top-10 country ordering changes only modestly between
"combined" and "blackout-only" views — see the table above. Full
sensitivity → `03_robustness.ipynb`.


In [22]:
def shutdown_day_rollup(frame: pd.DataFrame, view: str = "combined") -> pd.DataFrame:
    """Country-level shutdown-day totals under the chosen view."""
    f = frame.copy()
    f["bucket"] = (
        f["platform_block"].map({True: "platform_block"})
            .fillna(f["type"].astype("string"))
    )
    if view == "combined":
        sub = f
    elif view == "blackout_only":
        sub = f[f["bucket"] == "full_blackout"]
    elif view == "separated":
        return (
            f.groupby(["country", "iso3", "bucket"], dropna=False)["duration_days"]
             .sum(min_count=1).reset_index()
        )
    else:
        raise ValueError(f"unknown view: {view!r}")
    return (
        sub.groupby(["country", "iso3"], dropna=False)["duration_days"]
           .sum(min_count=1).reset_index()
           .sort_values("duration_days", ascending=False)
    )

# Sanity-check both views.
roll_combined = shutdown_day_rollup(df_view, view="combined")
roll_blackout = shutdown_day_rollup(df_view, view="blackout_only")
print(f"Top-5 combined:     {roll_combined.head(5)['country'].tolist()}")
print(f"Top-5 blackout-only: {roll_blackout.head(5)['country'].tolist()}")


Top-5 combined:     ['Myanmar', 'Iran (Islamic Republic of)', 'Russian Federation', 'Pakistan', 'Turkey']
Top-5 blackout-only: ['Myanmar', 'Pakistan', 'Bangladesh', 'Ethiopia', 'India']


## 10. Decision: snapshot pinning for revisable sources

**Problem.** Both Access Now's #KeepItOn registry and Top10VPN's annual
reports *restate prior years* as new evidence emerges. Access Now adds
events for past years that were not yet reported when the registry was
last captured; Top10VPN reissues its annual report each year with adjusted
historical inputs. Without pinning, the same notebook re-run six months
from now would produce different numbers — and we wouldn't know whether
that drift was the data improving or our pipeline regressing.

**Diagnostic.** What's the magnitude of revision between snapshots? This
is *retrospective* — the project has a single 2026-05-20 snapshot for each
source so the diagnostic is structurally a placeholder until a second
snapshot exists. We document the *strategy* now; the magnitude table will
populate the first time we re-snapshot.


In [23]:
# Placeholder: when a second snapshot exists, this block computes per-country
# revision magnitude. For now, log the schema we'll use so the next
# snapshot session can drop it straight in.
SNAPSHOT_REVISION_SCHEMA = [
    "country", "metric", "value_old_snapshot", "value_new_snapshot",
    "abs_revision", "pct_revision",
]
print("Snapshot revision schema (populated next snapshot):")
print(SNAPSHOT_REVISION_SCHEMA)
print()
print("Current pinned snapshots:")
print("  data/processed/access_now_snapshot_2026-05-20.parquet")
print("  data/processed/top10vpn_snapshot_2026-05-20.parquet")
print("Re-snapshot procedure: rerun notebooks/_scratch/snapshot_access_now.py")
print("with a new date, commit alongside the existing snapshot, then run this")
print("cell against both pinned files.")


Snapshot revision schema (populated next snapshot):
['country', 'metric', 'value_old_snapshot', 'value_new_snapshot', 'abs_revision', 'pct_revision']

Current pinned snapshots:
  data/processed/access_now_snapshot_2026-05-20.parquet
  data/processed/top10vpn_snapshot_2026-05-20.parquet
Re-snapshot procedure: rerun notebooks/_scratch/snapshot_access_now.py
with a new date, commit alongside the existing snapshot, then run this
cell against both pinned files.


**Options considered.**

- (a) **No pinning — re-fetch each run.** Notebook is non-reproducible
  across time. Two readers running the same code two months apart see
  different numbers.
- (b) **Pin only Access Now.** Half-pinned: Top10VPN's revisions still
  drift cost figures even when event data is stable.
- (c) **Pin both sources with dated parquets** (Access Now + Top10VPN),
  document the pin date, log revision magnitudes when re-snapshotting.

**Decision.** **(c) Pin both sources with dated parquet files.** Snapshots
live at `data/processed/{access_now,top10vpn}_snapshot_2026-05-20.parquet`
and are tracked under git. The notebook reads from these snapshots, *not*
from the live source. Re-snapshotting is an explicit, documented step that
appends a new dated parquet, leaves the prior parquets in place, and
populates the revision-magnitude table above.

**Rationale.** The project is a *retrospective* analysis (2019–latest)
that must be re-runnable in 6 / 12 months to produce identical numbers
absent an explicit re-snapshot decision. Without (c), the methodology
caveat about Top10VPN's debated figures would be compounded by silent
year-over-year drift in those same figures.

**Sensitivity.** Until a second snapshot exists, sensitivity to revisions
is structurally untestable. The expectation, based on Access Now's update
cadence and Top10VPN's annual cycle, is that the median per-country
revision will be small (<5%) for years more than 18 months in the past
and larger (5–25%) for the most recent year. The next snapshot session
fills in the actual numbers.


## 11. Persist the analytic dataset

S5 (viz) and S6 (dashboard) read the analytic dataset from this committed
parquet, *not* from the upstream snapshots. This isolates the analysis
plumbing from the viz layer and means the dashboard can be re-launched
without re-running the cleaning notebook.


In [24]:
from internet_shutdowns.data import write_processed

# Drop helper / scratch columns before persisting.
to_persist = analytic.copy()
out_path = write_processed(to_persist, "analytic_dataset_2026-05-20")
print(f"wrote {out_path.relative_to(out_path.parents[2])}")
print(f"  rows × cols: {to_persist.shape}")
print(f"  size: {out_path.stat().st_size / 1024:.1f} KB")


wrote data\processed\analytic_dataset_2026-05-20.parquet
  rows × cols: (1511, 66)
  size: 542.1 KB


## 12. Hero figure + viz showcase

Three reusable Plotly figures live in
`src/internet_shutdowns/viz.py`. They read the analytic dataset (the
contract from §11) and are called from both this notebook and the
Streamlit dashboard (S6) — one viz pipeline, two surfaces.

The hero figure (top-10 countries by total estimated shutdown cost) is
written to `figures/hero.png` at 800×800 @2x — the LinkedIn-thumbnail
constraint from `CLAUDE.md`. **The Top10VPN methodology caveat is baked
into the title and must remain legible at thumbnail size** — that's the
load-bearing UX decision for this figure.


In [25]:
import pandas as pd
from internet_shutdowns.viz import (
    METHODOLOGY_CAVEAT,
    save_figure,
    time_series,
    top10_bar,
    world_choropleth,
)

# Re-load from the persisted parquet so §12 is decoupled from §§1–11.
df_viz = pd.read_parquet("../data/processed/analytic_dataset_2026-05-20.parquet")
print(f"Loaded analytic dataset: {df_viz.shape[0]:,} events × {df_viz.shape[1]} cols")
print(f"Methodology caveat string: {METHODOLOGY_CAVEAT!r}")


Loaded analytic dataset: 1,511 events × 66 cols
Methodology caveat string: 'Top10VPN methodology — see limitations'


**Hero figure — top 10 countries by estimated shutdown cost.**

In [26]:
hero = top10_bar(df_viz, metric="total_cost_usd", year_range=(2019, 2025))
hero.show()


**World choropleth — total shutdown-days by country.**

In [27]:
world_choropleth(df_viz, metric="total_shutdown_days").show()


**Time series — monthly recorded events 2019–2025.**

In [28]:
time_series(df_viz, metric="event_count").show()


**Cumulative cost — same series, dollar-weighted.**

In [29]:
time_series(df_viz, metric="cumulative_cost").show()


### Save the hero figure

`figures/hero.png` is the static export for the README and social-media
share previews. 800×800 @2x means the saved PNG is 1600×1600 — sharp at
thumbnail size, and Plotly's title `<sup>` rendering keeps the caveat
legible.


In [30]:
hero_path = save_figure(hero, "hero", width=800, height=800, scale=2)
print(f"Saved {hero_path.relative_to(hero_path.parents[1])} "
      f"({hero_path.stat().st_size / 1024:.1f} KB)")


Saved figures\hero.png (135.1 KB)


---

*Session 6 (Streamlit dashboard) imports the same viz helpers and reads
the same analytic parquet. No re-cleaning, no parallel viz pipeline.*
